# Gli esperimenti sul dataset esteso

La Fase 1 ha stabilito due cose sul dataset del paper. Che il residuo dell'autoencoder
ordina i cuscinetti **per esemplare** e non per stato: la dispersione fra i diciassette
esemplari vale un fattore due, la differenza fra sano e guasto il 9,5%. E che l'unica
modifica che sposta davvero la separazione e portare il segnale dentro il campo della SELU
con una costante unica, che alza l'area sotto la curva da 0,600 a 0,775.

Qui si riprendono quelle due conclusioni e le si mette alla prova su dati piu grandi e con
una valutazione piu severa: ventinove cuscinetti invece di diciassette, quattro condizioni
operative invece di una, e soprattutto la divisione fra addestramento e verifica fatta per
**cuscinetti interi**. Nella replica gli esemplari di verifica erano gli stessi
dell'addestramento; qui il modello deve pronunciarsi su cuscinetti che non ha mai visto, che
e la situazione reale in fabbrica.

Il notebook e diviso in tre parti. La prima sceglie la configurazione dell'autoencoder. La
seconda misura i residui nei sette esperimenti previsti, e costa pochi minuti. La terza
completa la catena fino alla rete convolutiva, ed e la parte cara.

**Il protocollo di addestramento e unico per tutti gli esperimenti**, cosi le righe della
tabella finale si confrontano fra loro: cuscinetti sani di addestramento separati da quelli
di validazione, tetto di cinquecento epoche, arresto anticipato a pazienza venti con
ripristino dei pesi dell'epoca migliore. Nessun esperimento fa eccezione.

In [ ]:
!pip -q install scipy scikit-learn

In [ ]:
import os, sys, gc, time, json, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']
percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break
if percorso_codice is None:
    print('codice non trovato in locale, clono il repo')
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'], check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'
sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='05_esperimenti_esteso')
dev = f.dispositivo()
seme = 0

# quali esperimenti arrivano fino alla rete convolutiva. Ognuno costa mezz'ora
# scarsa: si parte dai due che rispondono alle domande principali, gli altri si
# aggiungono qui se il tempo lo consente.
CNN_DA_ESEGUIRE = ['soli_reali', 'artificiale_verso_reale']

print('codice da', percorso_codice)
print('dispositivo', dev)

## Il dataset

I frame stanno su Drive, che su Colab e un disco di rete: leggerne 3,5 GB a pezzi sparsi
sarebbe lentissimo. Si copiano quindi una volta sul disco locale della macchina e poi si
aprono in modalita mappata, cosi la memoria non se ne accorge e le letture sono veloci.

In [ ]:
sorgente = os.path.join(P['dataset'], 'frame.npy')
locale = '/content/frame.npy'
if not os.path.exists(locale):
    print('copio il dataset sul disco locale...')
    partenza = time.time()
    shutil.copy(sorgente, locale)
    print('copiato in', round(time.time() - partenza), 's')

X = np.load(locale, mmap_mode='r')
anagrafica = pd.read_csv(os.path.join(P['dataset'], 'anagrafica.csv'))
parametri = json.load(open(os.path.join(P['dataset'], 'parametri.json')))

print('frame:', X.shape, '| standardizzati per frame:',
      parametri['standardizzazione_per_frame'])
print('cuscinetti:', anagrafica['cuscinetto'].nunique(),
      '| registrazioni:', anagrafica['registrazione'].nunique())
print()
print(anagrafica.groupby('regime').size().to_string())
print()
print('righe dell anagrafica e frame della matrice coincidono:',
      len(anagrafica) == len(X))

## La scala del segnale

Il fattore si calcola come nella Fase 1: dal picco piu alto dei soli frame sani di
addestramento, portato a 1,5 per lasciare un margine sotto il pavimento della SELU.

C'e pero una differenza che va misurata prima di procedere. Nella replica c'era una sola
condizione operativa; qui ce ne sono quattro, e la corrente cambia molto da una all'altra.
Il troncamento della SELU dipende dall'ampiezza, quindi non agisce allo stesso modo
dappertutto.

E non e nemmeno garantito che la scala lo elimini ovunque. Il fattore nasce dal massimo dei
soli **sani di addestramento**, mentre il pavimento vale $-1{,}7581$: resta troncato tutto
cio che supera in modulo $1{,}172$ volte quel massimo. Sono diciassette punti percentuali di
margine, e cuscinetti guasti o regimi diversi possono superarli. La domanda va quindi chiusa
con un conteggio, non con un ragionamento, e il conteggio va fatto su **tutti** i frame del
dataset: guardarne una fetta iniziale significherebbe guardare i primi cuscinetti in ordine
di scrittura, non un campione rappresentativo.

In [ ]:
sani_dae = anagrafica['cuscinetto'].isin(config.SANI_DAE_TRAIN).values
frame_sani = np.asarray(X[np.flatnonzero(sani_dae)])
fattore, massimo = f.scala_globale(frame_sani)
del frame_sani
gc.collect()

soglia_ampere = abs(config.PAVIMENTO_SELU) / fattore
print('massimo assoluto dei sani di addestramento:', round(massimo, 4), 'A')
print('fattore di scala:', round(fattore, 6))
print('dopo la scala resta troncato cio che supera', round(soglia_ampere, 4), 'A,',
      'cioe', round(soglia_ampere / massimo, 3), 'volte quel massimo')
print()

# Conteggio su tutti i frame, letti a blocchi per non caricare 3,5 GB in memoria.
PASSO_LETTURA = 8192
regimi = sorted(anagrafica['regime'].unique())
regime_del_frame = anagrafica['regime'].values
conteggi = {r: {'frame': 0, 'campioni': 0, 'sotto': 0, 'sotto_dopo': 0,
                'somma_quadrati': 0.0, 'picco': 0.0} for r in regimi}

partenza = time.time()
for i in range(0, len(X), PASSO_LETTURA):
    blocco = np.asarray(X[i:i + PASSO_LETTURA], dtype=np.float32)
    etichette_blocco = regime_del_frame[i:i + PASSO_LETTURA]
    for r in regimi:
        maschera = etichette_blocco == r
        if not maschera.any():
            continue
        v = blocco[maschera]
        c = conteggi[r]
        c['frame'] += int(v.shape[0])
        c['campioni'] += int(v.size)
        c['sotto'] += int(np.count_nonzero(v < config.PAVIMENTO_SELU))
        c['sotto_dopo'] += int(np.count_nonzero(v * fattore < config.PAVIMENTO_SELU))
        c['somma_quadrati'] += float(np.sum(v.astype(np.float64) ** 2))
        c['picco'] = max(c['picco'], float(np.max(np.abs(v))))
print('scansione di', len(X), 'frame in', round(time.time() - partenza), 's')
print()

righe = []
for r in regimi:
    c = conteggi[r]
    righe.append({'regime': r,
                  'rpm': config.REGIMI[r]['rpm'],
                  'coppia_Nm': config.REGIMI[r]['coppia_Nm'],
                  'forza_N': config.REGIMI[r]['forza_N'],
                  'frame': c['frame'],
                  'rms': float(np.sqrt(c['somma_quadrati'] / c['campioni'])),
                  'picco': c['picco'],
                  'sotto_pavimento_pct': 100 * c['sotto'] / c['campioni'],
                  'sotto_dopo_scala_pct': 100 * c['sotto_dopo'] / c['campioni']})
tabella_regimi = pd.DataFrame(righe)

totale = {'regime': 'tutti', 'rpm': np.nan, 'coppia_Nm': np.nan, 'forza_N': np.nan,
          'frame': int(sum(c['frame'] for c in conteggi.values())),
          'rms': float(np.sqrt(sum(c['somma_quadrati'] for c in conteggi.values())
                               / sum(c['campioni'] for c in conteggi.values()))),
          'picco': max(c['picco'] for c in conteggi.values()),
          'sotto_pavimento_pct': 100 * sum(c['sotto'] for c in conteggi.values())
                                 / sum(c['campioni'] for c in conteggi.values()),
          'sotto_dopo_scala_pct': 100 * sum(c['sotto_dopo'] for c in conteggi.values())
                                  / sum(c['campioni'] for c in conteggi.values())}

print(pd.concat([tabella_regimi, pd.DataFrame([totale])],
                ignore_index=True).round(3).to_string(index=False))
print()
print('frame esaminati:', totale['frame'], 'su', len(X), '- e tutto il dataset:',
      totale['frame'] == len(X))
print('la scala azzera il troncamento su tutti i regimi:',
      bool(tabella_regimi['sotto_dopo_scala_pct'].max() == 0.0))

## Come si misura un esperimento

Ogni esperimento addestra un autoencoder sui soli cuscinetti sani che il protocollo gli
assegna, poi calcola il residuo su un insieme di verifica e lo misura sempre allo stesso
modo: residuo medio per classe, rapporto sul sano, area sotto la curva, e il dettaglio per
singolo esemplare, che nella Fase 1 si e rivelato il numero piu informativo.

L'autoencoder usa la catena del paper adattata all'ingresso da 4273 campioni. Il protocollo
di addestramento e **lo stesso in tutti gli esperimenti**: cuscinetti di validazione
separati da quelli di addestramento, tetto di cinquecento epoche, arresto anticipato a
pazienza venti. La Fase 1 ha mostrato che l'arresto anticipato da lo stesso risultato delle
cinquecento epoche fisse, dimezzando il tempo.

Gli autoencoder identici vengono addestrati **una volta sola**. Diversi esperimenti
condividono gli stessi cuscinetti sani, lo stesso seme e gli stessi parametri: senza una
memoria il notebook riaddestrerebbe sei volte lo stesso modello, con lo stesso risultato.
Una conseguenza da tenere a mente nella lettura della tabella finale: le righe che
condividono l'autoencoder e l'insieme di verifica sono identiche **per costruzione**, non
per coincidenza.

In [ ]:
DIMENSIONI = [config.LUNGHEZZA_GIRO, 1280, 640, 320, 128, 32,
              128, 320, 640, 1280, config.LUNGHEZZA_GIRO]
PAZIENZA = 20
EPOCHE = 500

_memoria_modelli = {}


def indici(cuscinetti, regimi=None):
    """Posizioni dei frame che appartengono a quei cuscinetti e a quei regimi."""
    m = anagrafica['cuscinetto'].isin(cuscinetti).values
    if regimi is not None:
        m = m & anagrafica['regime'].isin(regimi).values
    return np.flatnonzero(m)


def addestra(cuscinetti_train, cuscinetti_val, regimi=None, scala=True,
             epoche=EPOCHE, pazienza=PAZIENZA, etichetta=''):
    """
    Autoencoder sui soli sani indicati. Restituisce (modello, fattore, curve).

    Un modello gia addestrato con gli stessi identici argomenti viene riusato: il
    seme e fissato, quindi riaddestrarlo darebbe gli stessi pesi e gli stessi numeri.
    """
    chiave = (tuple(cuscinetti_train), tuple(cuscinetti_val),
              tuple(regimi) if regimi is not None else None, scala, epoche, pazienza)
    if chiave in _memoria_modelli:
        modello, k, curve = _memoria_modelli[chiave]
        print(etichetta, '| autoencoder identico gia addestrato, lo riuso |',
              len(curve[1]), 'epoche')
        return modello, k, curve

    k = 1.0
    tr = np.asarray(X[indici(cuscinetti_train, regimi)], dtype=np.float32)
    if scala:
        k, _ = f.scala_globale(tr)
        tr = tr * k
    va = np.asarray(X[indici(cuscinetti_val, regimi)], dtype=np.float32) * k

    print(etichetta, '| addestramento', tr.shape[0], 'frame su',
          len(cuscinetti_train), 'cuscinetti | validazione', va.shape[0], 'frame su',
          len(cuscinetti_val))
    modello, c_tr, c_va = f.addestra_dae(tr, va, uscita_selu=True, dimensioni=DIMENSIONI,
                                         epoche=epoche, lotto=256, passo=3e-4,
                                         pazienza=pazienza, seme=seme, dev=dev,
                                         stampa_ogni=25)
    del tr, va
    gc.collect()
    _memoria_modelli[chiave] = (modello, k, (c_tr, c_va))
    return modello, k, (c_tr, c_va)


def misura(modello, k, cuscinetti, regimi=None, etichetta='', curve=None,
           addestramento=None, validazione=None):
    """Residuo per classe, rapporti, AUC e dettaglio per esemplare."""
    posizioni = indici(cuscinetti, regimi)
    parte = anagrafica.iloc[posizioni]
    mse = np.empty(len(posizioni), dtype=np.float64)
    for i in range(0, len(posizioni), 2048):
        blocco = np.asarray(X[posizioni[i:i + 2048]], dtype=np.float32) * k
        residuo = f.calcola_residui(modello, blocco, dev=dev) / k
        mse[i:i + len(blocco)] = f.mse_per_frame(residuo)

    classi = parte['classe'].values
    per_classe = [float(np.mean(mse[classi == c])) if (classi == c).any() else np.nan
                  for c in (0, 1, 2)]
    auc = f.auc_residuo(mse, classi)
    dettaglio = f.residuo_per_cuscinetto(mse, parte['cuscinetto'].values, classi)

    riga = {'esperimento': etichetta,
            'addestramento': ' '.join(addestramento) if addestramento else '',
            'validazione': ' '.join(validazione) if validazione else '',
            'regimi_addestramento': 'tutti' if regimi is None else ' '.join(regimi),
            'epoche': len(curve[1]) if curve else np.nan,
            'epoca_minimo': int(np.argmin(curve[1])) + 1 if curve else np.nan,
            'verifica': ' '.join(sorted(parte['cuscinetto'].unique())),
            'n_cuscinetti_verifica': int(parte['cuscinetto'].nunique()),
            'n_frame_verifica': int(len(posizioni)),
            'regimi_verifica': 'tutti' if regimi is None else ' '.join(regimi),
            'residuo_sano': per_classe[0],
            'residuo_esterno': per_classe[1],
            'residuo_interno': per_classe[2],
            'rapporto_esterno': per_classe[1] / per_classe[0],
            'rapporto_interno': per_classe[2] / per_classe[0],
            'auc_esterno': auc['esterno'], 'auc_interno': auc['interno'],
            'auc_media': auc['media'],
            'dispersione_esemplari': float(dettaglio['media'].max()
                                           / dettaglio['media'].min())}

    print('  ', etichetta, '|', len(posizioni), 'frame di verifica su',
          parte['cuscinetto'].nunique(), 'cuscinetti')
    print('   rapporti {:.3f} / {:.3f}   AUC {:.4f}   dispersione fra esemplari {:.2f}'.format(
        riga['rapporto_esterno'], riga['rapporto_interno'],
        riga['auc_media'], riga['dispersione_esemplari']))
    return riga, dettaglio, mse


risultati = []
dettagli = {}
print('protocollo comune: tetto', EPOCHE, 'epoche, pazienza', PAZIENZA,
      ', collo di bottiglia', DIMENSIONI[5])

## Parte prima. La scala serve anche qui?

La Fase 1 ha trovato nella scala globale l'unica modifica che sposta la separazione. Ma la
Fase 1 aveva un regime solo, dove il troncamento colpiva un quarto dei campioni. Qui i
regimi sono quattro e il troncamento non e uniforme, quindi la domanda va rifatta.

Si addestrano due autoencoder identici in tutto tranne la scala del segnale, sugli stessi
tre cuscinetti sani e su tutti e quattro i regimi, e si misurano sui cuscinetti a danno
reale, che nessuno dei due ha mai visto.

Una avvertenza sulla lettura del risultato. Le due configurazioni vengono addestrate una
volta ciascuna, a seme fissato. La Fase 1 ha misurato la dispersione dei classificatori
statistici su cento suddivisioni e ha trovato scarti tipo fra 2,4 e 2,9 punti; Vieira et al.
riportano sullo stesso dataset AUROC con deviazione fra 13 e 19 punti. Una differenza fra le
due configurazioni piu piccola di cosi non e distinguibile dal rumore con una sola
esecuzione, e va letta come tale.

In [ ]:
verifica_configurazioni = config.REALI[1] + config.REALI[2] + config.SANI_TEST
curve_parte1 = {}

for etichetta, scala in [('segnale non scalato', False), ('scala globale', True)]:
    modello, k, curve = addestra(config.SANI_DAE_TRAIN, config.SANI_DAE_VAL,
                                 scala=scala, etichetta=etichetta)
    riga, dettaglio, _ = misura(modello, k, verifica_configurazioni, etichetta=etichetta,
                                curve=curve, addestramento=config.SANI_DAE_TRAIN,
                                validazione=config.SANI_DAE_VAL)
    riga['scala'] = 'si' if scala else 'no'
    risultati.append(riga)
    dettagli[etichetta] = dettaglio
    curve_parte1[etichetta] = curve
    gc.collect()
    print()

for etichetta, (c_tr, c_va) in curve_parte1.items():
    print('{:22s} {:3d} epoche, minimo all epoca {:3d}, errore {:.6f}'.format(
        etichetta, len(c_va), int(np.argmin(c_va)) + 1, min(c_va)))

## Parte seconda. I sette esperimenti, misurati sui residui

Ogni esperimento cambia una cosa sola rispetto agli altri, e ciascuno risponde a una domanda
che la replica non poteva porsi.

**Soli danni reali** usa gli stessi cuscinetti del paper ma con gli esemplari separati fra
addestramento e verifica. Non va chiamato replica: i regimi sono quattro e la divisione e
piu severa. La verifica cade su tre soli cuscinetti, uno per classe, ed e il numero piu
fragile della tabella.

**Artificiale verso reale** e l'esperimento principale del dataset esteso: si impara su
danni fatti a macchina, che costano poco, e si verifica su danni cresciuti in prove di vita
accelerate, che costano molto. E la domanda della manutenzione predittiva. Attenzione pero:
a livello di **residuo** questo esperimento non ha niente di artificiale, perche
l'autoencoder vede soltanto cuscinetti sani. E la rete convolutiva della Parte terza a
imparare sui guasti finti, quindi e li che l'etichetta ha senso.

**Velocita** e **coppia** chiedono se il residuo rappresenti il guasto oppure il punto di
funzionamento. Entrambi addestrano su una condizione sola e verificano su quella e su
un'altra. Sono costruiti in modo da cambiare **una variabile per volta**: per la velocita si
confrontano 1500 e 900 giri a coppia piena, per la coppia si confrontano coppia piena e
ridotta a 1500 giri. Il piano originale prevedeva di addestrare sui tre regimi a 1500 giri,
ma quelli differiscono anche per coppia e i due effetti resterebbero confusi.

**Severita** addestra sui danni artificiali estesi e verifica su quelli incipienti.

**Rodaggio** sta in una cella a parte perche ha un'assegnazione dei cuscinetti tutta sua.

In [ ]:
UN_REGIME = ['N15_M07_F10']

esperimenti = [
    ('soli reali',            config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, None,
     config.INSIEMI['soli_reali']['test'], None),
    ('artificiale -> reale',  config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, None,
     config.INSIEMI['artificiale_verso_reale']['test'], None),
    ('severita',              config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, None,
     config.INSIEMI['severita']['test'], None),
    ('velocita, 1500 giri',   config.SANI_DAE_TRAIN, config.SANI_DAE_VAL, UN_REGIME,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, UN_REGIME),
    ('velocita, 900 giri',    None, None, None,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, ['N09_M07_F10']),
    ('coppia piena',          None, None, None,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, UN_REGIME),
    ('coppia ridotta',        None, None, None,
     config.REALI[1] + config.REALI[2] + config.SANI_TEST, ['N15_M01_F10']),
]

modello_corrente, k_corrente, curve_corrente = None, 1.0, None
train_corrente, val_corrente, regimi_corrente = None, None, None

for etichetta, tr, va, reg_tr, test, reg_test in esperimenti:
    if tr is not None:                      # None: riusa l'autoencoder precedente
        modello_corrente, k_corrente, curve_corrente = addestra(
            tr, va, regimi=reg_tr, etichetta=etichetta)
        train_corrente, val_corrente, regimi_corrente = tr, va, reg_tr
    riga, dettaglio, _ = misura(modello_corrente, k_corrente, test, regimi=reg_test,
                                etichetta=etichetta, curve=curve_corrente,
                                addestramento=train_corrente, validazione=val_corrente)
    riga['regimi_addestramento'] = ('tutti' if regimi_corrente is None
                                    else ' '.join(regimi_corrente))
    riga['regimi_verifica'] = 'tutti' if reg_test is None else ' '.join(reg_test)
    riga['scala'] = 'si'
    risultati.append(riga)
    dettagli[etichetta] = dettaglio
    print()

### Rodaggio, cioe quanto costa chiamare "normale" un cuscinetto usato

Le schede dichiarano per i sei sani ore di rodaggio molto diverse, da una a oltre cinquanta.
La domanda e descrittiva e riguarda il modo di fallire piu dannoso per un sistema di
monitoraggio: **un cuscinetto integro ma con superfici piu assestate produce un residuo piu
alto?** Se si, il sistema segnala guasti che non ci sono, e un sistema che grida al lupo
viene spento.

L'autoencoder impara quindi da cuscinetti **poco rodati** e il residuo si misura su quelli
**molto rodati**, con gli undici danni reali accanto come termine di paragone. Se K001 e
K002, che sono sani, finiscono in mezzo o sopra ai guasti nella graduatoria dei residui,
l'area sotto la curva scende sotto 0,5 e il falso allarme e misurato invece che ipotizzato.

L'assegnazione dei sei sani e stretta, perche ognuno ha un ruolo e nessuno puo averne due:

| cuscinetto | ore | ruolo |
|---|---|---|
| K003 | 1 | addestramento |
| K005 | 10 | addestramento |
| K004 | 5 | validazione, la stessa di tutti gli altri esperimenti |
| K002 | 19 | misura del residuo |
| K001 | oltre 50 | misura del residuo |
| K006 | 16 | sigillato per la verifica finale |

L'autoencoder impara dunque da **due** cuscinetti invece che da tre. E il prezzo per tenere
qui lo stesso protocollo degli altri esperimenti — validazione su un cuscinetto intero e
separato, arresto anticipato a pazienza venti — invece di fissare un numero di epoche a
mano. Il costo e piccolo e gia misurato: nella Fase 1 la variante con il 44% di frame in
meno ha ottenuto l'errore di ricostruzione **piu basso** di tutte, perche conta l'omogeneita
dei sani, non la loro quantita.

Conseguenza da dichiarare: il valore **assoluto** del residuo di questa riga non si confronta
con le altre, che vengono da un modello addestrato su tre cuscinetti. La domanda del
rodaggio e pero interna, cioe dove cadono K001 e K002 rispetto ai guasti dentro questo
modello, e a quella la riga risponde per intero.

In [ ]:
SANI_RODAGGIO_TRAIN = ['K003', 'K005']          # 1 h e 10 h, entrambi poco rodati
SANI_RODAGGIO_VAL = config.SANI_DAE_VAL         # K004, 5 h: la validazione di sempre
sani_misura = [b for b in config.CUSCINETTI_PER_CLASSE[0]
               if b not in SANI_RODAGGIO_TRAIN + SANI_RODAGGIO_VAL + config.SANI_TEST]
verifica_rodaggio = config.REALI[1] + config.REALI[2] + sani_misura

descrivi = lambda elenco: ' '.join('{} ({:g} h)'.format(b, config.ORE_RODAGGIO[b])
                                   for b in elenco)
print('addestramento :', descrivi(SANI_RODAGGIO_TRAIN))
print('validazione   :', descrivi(SANI_RODAGGIO_VAL))
print('misura        :', descrivi(sani_misura), '+ gli undici danni reali')
print('sigillato     :', descrivi(config.SANI_TEST))
assert not set(SANI_RODAGGIO_TRAIN) & set(SANI_RODAGGIO_VAL + config.SANI_TEST + sani_misura)
print()

modello, k, curve = addestra(SANI_RODAGGIO_TRAIN, SANI_RODAGGIO_VAL, etichetta='rodaggio')
riga, dettaglio, _ = misura(modello, k, verifica_rodaggio, etichetta='rodaggio',
                            curve=curve, addestramento=SANI_RODAGGIO_TRAIN,
                            validazione=SANI_RODAGGIO_VAL)
riga['scala'] = 'si'
risultati.append(riga)
dettagli['rodaggio'] = dettaglio
rodaggio_dettaglio = dettaglio
gc.collect()
print()

graduatoria = dettaglio.copy()
graduatoria['posizione'] = np.arange(1, len(graduatoria) + 1)
graduatoria['ore'] = graduatoria['cuscinetto'].map(config.ORE_RODAGGIO)
print('graduatoria dei residui, dal piu basso al piu alto:')
print(graduatoria[['posizione', 'cuscinetto', 'classe', 'ore', 'n', 'media']]
      .to_string(index=False, na_rep=''))
print()
for b in sani_misura:
    posto = int(graduatoria.loc[graduatoria['cuscinetto'] == b, 'posizione'].iloc[0])
    print('{} e sano con {:g} h di rodaggio e sta al posto {} su {}: ha residuo piu alto di'
          ' {} cuscinetti guasti'.format(b, config.ORE_RODAGGIO[b], posto,
                                         len(graduatoria), posto - 1))

## Il quadro d'insieme dei residui

Due tabelle invece di una. La prima dichiara **come** e stato ottenuto ogni numero, cioe con
quali cuscinetti e per quante epoche: senza quella colonna le righe non sono confrontabili e
il lettore non ha modo di accorgersene. La seconda riporta i risultati.

In [ ]:
confronto = pd.DataFrame(risultati)

protocollo = ['esperimento', 'addestramento', 'validazione', 'regimi_addestramento',
              'scala', 'epoche', 'epoca_minimo', 'n_cuscinetti_verifica',
              'n_frame_verifica', 'regimi_verifica']
esiti = ['esperimento', 'residuo_sano', 'rapporto_esterno', 'rapporto_interno',
         'auc_esterno', 'auc_interno', 'auc_media', 'dispersione_esemplari']

print('COME sono stati ottenuti')
print(confronto[protocollo].to_string(index=False))
print()
print('COSA e uscito')
print(confronto[esiti].round(4).to_string(index=False))
print()
print('cuscinetti di verifica, riga per riga')
print(confronto[['esperimento', 'verifica']].to_string(index=False))
print()
print('riferimenti:')
print('   Fase 1, dataset del paper con la scala globale: rapporti 1,414 / 1,356,'
      ' AUC media 0,775')
print('   valori dichiarati dal paper: rapporti 3,712 / 4,606')
print('   caso puro: AUC 0,500. Sotto 0,5 l ordinamento e rovesciato.')
print()
sotto_meta = confronto[confronto['auc_media'] < 0.5]['esperimento'].tolist()
print('esperimenti con AUC media sotto il caso puro:',
      ', '.join(sotto_meta) if sotto_meta else 'nessuno')

## Parte terza. La rete convolutiva

Qui la catena si chiude. Il residuo dei blocchi da quattordici giri diventa l'ingresso della
rete, che viene addestrata sui cuscinetti previsti dal protocollo e verificata su esemplari
mai visti. I quattordici giri non sono un numero tondo scelto a caso: le frequenze di guasto
non sono multipli interi della rotazione e il loro schema si ripete per intero ogni 14,2
giri, quindi sotto quella soglia la cadenza del difetto non ha modo di manifestarsi.

Il conteggio viene fatto su tre unita diverse a partire dalle stesse previsioni. Sul
**blocco**, che e quello che riporta il paper ed e il piu ottimistico, perche blocchi della
stessa registrazione sono quasi copie l'uno dell'altro. Sulla **registrazione**, per voto di
maggioranza fra i suoi blocchi. E sul **cuscinetto**, che e l'unico che risponde alla domanda
industriale: quanti esemplari sono stati diagnosticati correttamente.

Accanto a ogni accuratezza si riporta quella del **classificatore degenere**, cioe di chi
predice sempre la classe piu numerosa senza guardare i dati. Senza quel numero un 40% sembra
un risultato modesto ma positivo, mentre puo essere inferiore a non fare niente. Si riporta
anche il macro-F1 del degenere, perche accuratezza e macro-F1 possono dare verdetti opposti
e dichiararne uno solo sarebbe una scelta di comodo.

In [ ]:
def blocchi_di(cuscinetti, regimi=None):
    """Indici dei blocchi e loro anagrafica, per un insieme di cuscinetti."""
    m = anagrafica['cuscinetto'].isin(cuscinetti).values
    if regimi is not None:
        m = m & anagrafica['regime'].isin(regimi).values
    return f.costruisci_blocchi(anagrafica[m], config.GIRI_PER_BLOCCO)


def residui_blocchi(modello, k, indici_blocchi):
    """
    Residuo di ogni blocco, riportato in ampere.

    Non si usa f.residui_a_blocchi perche quella non conosce il fattore di scala:
    qui il segnale entra scalato e il residuo esce diviso per lo stesso fattore,
    cosi resta in ampere ed e confrontabile con le altre misure.
    """
    n, giri = indici_blocchi.shape
    uscita = np.empty((n, giri * X.shape[1]), dtype=np.float32)
    for i in range(0, n, 16):
        gruppo = indici_blocchi[i:i + 16]
        pezzo = np.asarray(X[gruppo.ravel()], dtype=np.float32) * k
        residuo = f.calcola_residui(modello, pezzo, dev=dev) / k
        uscita[i:i + len(gruppo)] = residuo.reshape(len(gruppo), -1)
    return uscita


esiti_cnn = {}
for nome in CNN_DA_ESEGUIRE:
    ins = config.INSIEMI[nome]
    print()
    print('===', nome)
    modello_dae, k, _ = addestra(ins['dae_train'], ins['dae_val'], etichetta='DAE di ' + nome)

    insiemi = {}
    for parte in ['train', 'val', 'test']:
        if len(ins[parte]) == 0:
            continue
        idx, ana = blocchi_di(ins[parte])
        insiemi[parte] = (residui_blocchi(modello_dae, k, idx),
                          ana['classe'].values.astype(np.int64), ana)
        print('   ', parte, len(idx), 'blocchi da', idx.shape[1], 'giri su',
              ana['cuscinetto'].nunique(), 'cuscinetti')
    gc.collect()

    if 'val' not in insiemi:                 # severita: nessuna validazione per costruzione
        insiemi['val'] = insiemi['train']
        pazienza_cnn = None
    else:
        pazienza_cnn = PAZIENZA

    modello_cnn, storia = f.addestra_cnn_insiemi(
        insiemi['train'][0], insiemi['train'][1],
        insiemi['val'][0], insiemi['val'][1],
        epoche=EPOCHE, lotto=64, passo=3e-4, pazienza=pazienza_cnn,
        seme=seme, dev=dev, stampa_ogni=10, etichetta=nome)

    previsioni = f.prevedi_cnn(modello_cnn, insiemi['test'][0], dev=dev)
    conteggio = f.conteggio_tre_livelli(previsioni, insiemi['test'][2])
    metriche = f.metriche(insiemi['test'][1], previsioni)

    # il classificatore degenere: predice sempre la classe piu numerosa
    vere_blocchi = insiemi['test'][1]
    ana_test = insiemi['test'][2]
    degenere = {'blocchi': f.baseline_degenere(vere_blocchi)}
    for livello, colonna in [('registrazioni', 'registrazione'),
                             ('cuscinetti', 'cuscinetto')]:
        degenere[livello] = f.baseline_degenere(
            ana_test.groupby(colonna)['classe'].first().values)
    metriche_degenere = f.metriche(
        vere_blocchi, np.full_like(vere_blocchi, degenere['blocchi']['classe']))

    print()
    for livello in ['blocchi', 'registrazioni', 'cuscinetti']:
        c = conteggio[livello]
        print(f"   {livello:14s} {c['giusti']:5d}/{c['totale']:<5d} = "
              f"{100 * c['accuratezza']:.2f} %")
    print()
    print('   classificatore degenere, che predice sempre',
          config.NOMI_CLASSI[degenere['blocchi']['classe']], ':')
    for livello in ['blocchi', 'registrazioni', 'cuscinetti']:
        print(f"      {livello:14s} {100 * degenere[livello]['accuratezza']:6.2f} %"
              f"   contro {100 * conteggio[livello]['accuratezza']:6.2f} % della rete")
    print(f"      {'macro-F1':14s} {metriche_degenere['f1']:6.3f}"
          f"     contro {metriche['f1']:6.3f} della rete")
    print()
    print(metriche['report'])
    print(pd.DataFrame(metriche['confusione'], index=config.NOMI_CLASSI,
                       columns=config.NOMI_CLASSI).to_string())
    print()
    dettaglio = conteggio['dettaglio_cuscinetti'].copy()
    dettaglio['classe_vera'] = [config.NOMI_CLASSI[c] for c in dettaglio['vera']]
    dettaglio['classe_prevista'] = [config.NOMI_CLASSI[c] for c in dettaglio['prevista']]
    dettaglio['esito'] = np.where(dettaglio['vera'] == dettaglio['prevista'],
                                  'giusto', 'sbagliato')
    print(dettaglio[['cuscinetto', 'classe_vera', 'classe_prevista', 'esito',
                     'blocchi', 'quota_voto']].to_string(index=False))

    degenere['macro_f1'] = metriche_degenere['f1']
    esiti_cnn[nome] = {'conteggio': conteggio, 'metriche': metriche, 'storia': storia,
                       'dettaglio': dettaglio, 'degenere': degenere}
    insiemi.clear()
    del insiemi, modello_cnn, modello_dae
    gc.collect()

## Riepilogo della rete convolutiva

In [ ]:
righe = []
for nome, e in esiti_cnn.items():
    riga = {'esperimento': nome}
    for livello in ['blocchi', 'registrazioni', 'cuscinetti']:
        riga[livello] = 100 * e['conteggio'][livello]['accuratezza']
        riga[livello + '_totale'] = e['conteggio'][livello]['totale']
        riga['degenere_' + livello] = 100 * e['degenere'][livello]['accuratezza']
    riga['macro_f1'] = e['metriche']['f1']
    riga['macro_f1_degenere'] = e['degenere']['macro_f1']
    riga['epoche'] = e['storia']['epoche']
    riga['epoca_minimo'] = e['storia']['epoca_minimo']
    riga['minuti'] = e['storia']['durata_s'] / 60
    righe.append(riga)

tabella_cnn = pd.DataFrame(righe)
if len(tabella_cnn):
    print('accuratezze, in percentuale, con accanto il classificatore degenere')
    print(tabella_cnn[['esperimento', 'blocchi', 'degenere_blocchi',
                       'registrazioni', 'degenere_registrazioni',
                       'cuscinetti', 'degenere_cuscinetti']]
          .round(2).to_string(index=False))
    print()
    print('macro-F1 e costo')
    print(tabella_cnn[['esperimento', 'macro_f1', 'macro_f1_degenere',
                       'epoche', 'epoca_minimo', 'minuti']].round(3).to_string(index=False))
    print()
    for _, r in tabella_cnn.iterrows():
        verso_blocchi = 'sopra' if r['blocchi'] > r['degenere_blocchi'] else 'SOTTO'
        verso_f1 = 'sopra' if r['macro_f1'] > r['macro_f1_degenere'] else 'SOTTO'
        print('{}: sui blocchi la rete sta {} il degenere, sul macro-F1 {}.'.format(
            r['esperimento'], verso_blocchi, verso_f1))
    print()
    print('per confronto, la replica del Capitolo 6 con la suddivisione permeabile del'
          ' paper: 46,86 % sui segmenti, e il paper ne dichiara 99,60')

## Le figure per la relazione

Cinque figure, tutte salvate in PNG a fondo bianco per la compilazione della tesi.

In [ ]:
from matplotlib.ticker import FuncFormatter

CLASSE_COLORE = {'normale': f.COLORI['normale'], 'esterno': f.COLORI['esterno'],
                 'interno': f.COLORI['interno']}

# la relazione e in italiano: sugli assi ci va la virgola, non il punto
VIRGOLA = FuncFormatter(lambda v, _: '{:g}'.format(v).replace('.', ','))


def barre_per_esemplare(ax, dettaglio, evidenzia=(), titolo=''):
    """Residuo medio di ogni esemplare, in ordine crescente e colorato per classe."""
    d = dettaglio.sort_values('media')
    posizioni = np.arange(len(d))
    colori = [CLASSE_COLORE.get(c, f.COLORI['neutro']) for c in d['classe']]
    ax.barh(posizioni, d['media'], color=colori)
    etichette = []
    for nome in d['cuscinetto']:
        etichette.append(nome + '  <' if nome in evidenzia else nome)
    ax.set_yticks(posizioni)
    ax.set_yticklabels(etichette, fontsize=8)
    for pos, nome in zip(posizioni, d['cuscinetto']):
        if nome in evidenzia:
            ax.get_yticklabels()[pos].set_fontweight('bold')
    ax.set_xlabel('residuo medio (A$^2$)')
    ax.set_title(titolo, fontsize=10)
    ax.xaxis.set_major_formatter(VIRGOLA)
    for classe, colore in CLASSE_COLORE.items():
        if (d['classe'] == classe).any():
            ax.plot([], [], 's', color=colore, label=classe)
    ax.legend(fontsize=7, loc='lower right')
    return d


# --- Figura 1. Il troncamento della SELU, regime per regime
fig, ax = plt.subplots(figsize=(9, 3.8))
posizioni = np.arange(len(tabella_regimi))
larghezza = 0.38
ax.bar(posizioni - larghezza / 2, tabella_regimi['sotto_pavimento_pct'], larghezza,
       color=f.COLORI['accento'], label='segnale originale')
ax.bar(posizioni + larghezza / 2, tabella_regimi['sotto_dopo_scala_pct'], larghezza,
       color=f.COLORI['nostro'], label='dopo la scala globale')
tetto = max(1.0, float(tabella_regimi['sotto_pavimento_pct'].max()))
for x, prima, dopo in zip(posizioni, tabella_regimi['sotto_pavimento_pct'],
                          tabella_regimi['sotto_dopo_scala_pct']):
    ax.text(x - larghezza / 2, prima + 0.02 * tetto,
            '{:.2f}'.format(prima).replace('.', ','), ha='center', fontsize=7)
    ax.text(x + larghezza / 2, dopo + 0.02 * tetto,
            '{:.2f}'.format(dopo).replace('.', ','), ha='center', fontsize=7)
ax.set_ylim(0, tetto * 1.18)
ax.yaxis.set_major_formatter(VIRGOLA)
ax.set_xticks(posizioni)
ax.set_xticklabels(['{}\n{} rpm, {} Nm, {} N'.format(r['regime'], r['rpm'],
                                                     r['coppia_Nm'], r['forza_N'])
                    for _, r in tabella_regimi.iterrows()], fontsize=7)
ax.set_ylabel('campioni sotto il pavimento (%)')
ax.set_title('Quanto segnale la SELU non puo rappresentare, per condizione operativa'
             ' (tutti i {} frame)'.format(len(X)), fontsize=10)
ax.legend(fontsize=8, loc='upper right')
f.salva_figura(fig, 'troncamento_per_regime', P['figure'])
plt.show()


# --- Figura 2. Le curve dei due autoencoder della Parte prima
fig, ax = plt.subplots(figsize=(9, 3.8))
for etichetta, colore in [('segnale non scalato', f.COLORI['accento']),
                          ('scala globale', f.COLORI['nostro'])]:
    if etichetta in curve_parte1:
        c_tr, c_va = curve_parte1[etichetta]
        ax.plot(range(1, len(c_va) + 1), c_va, color=colore, lw=1.2, label=etichetta)
        ax.plot(np.argmin(c_va) + 1, min(c_va), 'o', color=colore, ms=5)
ax.set_yscale('log')
ax.set_xlabel('epoca')
ax.set_ylabel('errore quadratico medio in validazione')
ax.set_title('Addestramento dei due autoencoder, arresto anticipato a pazienza'
             ' {}'.format(PAZIENZA), fontsize=10)
ax.legend(fontsize=8)
f.salva_figura(fig, 'curve_dae_scala', P['figure'])
plt.show()


# --- Figura 3. Il confronto fra gli esperimenti
fig, assi = plt.subplots(1, 2, figsize=(12, 4.6))

ordine = confronto.sort_values('auc_media')
posizioni = np.arange(len(ordine))
colori = [f.COLORI['nostro'] if v >= 0.5 else f.COLORI['accento']
          for v in ordine['auc_media']]
assi[0].barh(posizioni, ordine['auc_media'], color=colori)
assi[0].axvline(0.5, color=f.COLORI['scuro'], ls='-', lw=1)
assi[0].axvline(0.775, color=f.COLORI['neutro'], ls='--', lw=1,
                label='Fase 1, dataset del paper')
for y, v in zip(posizioni, ordine['auc_media']):
    assi[0].text(v + 0.008, y, '{:.3f}'.format(v).replace('.', ','),
                 va='center', fontsize=7)
assi[0].set_yticks(posizioni)
assi[0].set_yticklabels(ordine['esperimento'], fontsize=8)
assi[0].set_xlim(0, 1.05)
assi[0].xaxis.set_major_formatter(VIRGOLA)
assi[0].set_xlabel('AUC media')
assi[0].set_title('Separazione dei residui. A sinistra della linea nera\n'
                  'l ordinamento e rovesciato', fontsize=10)
assi[0].legend(fontsize=7, loc='lower right')

assi[1].barh(posizioni, ordine['dispersione_esemplari'], color=f.COLORI['interno'])
assi[1].axvline(1.0, color=f.COLORI['scuro'], ls='-', lw=1)
for y, v in zip(posizioni, ordine['dispersione_esemplari']):
    assi[1].text(v * 1.01, y, '{:.2f}'.format(v).replace('.', ','),
                 va='center', fontsize=7)
assi[1].set_yticks(posizioni)
assi[1].set_yticklabels([])
assi[1].xaxis.set_major_formatter(VIRGOLA)
assi[1].set_xlabel('residuo massimo diviso residuo minimo, fra esemplari')
assi[1].set_title('Quanto il residuo dipende dal singolo cuscinetto', fontsize=10)

f.salva_figura(fig, 'confronto_esperimenti', P['figure'])
plt.show()


# --- Figura 4. Il residuo esemplare per esemplare, nell esperimento principale
fig, assi = plt.subplots(1, 2, figsize=(12, 4.6))
barre_per_esemplare(assi[0], dettagli['artificiale -> reale'],
                    evidenzia=config.SANI_TEST,
                    titolo='Artificiale verso reale: residuo per esemplare')
barre_per_esemplare(assi[1], rodaggio_dettaglio, evidenzia=sani_misura,
                    titolo='Rodaggio: i due sani molto rodati fra i guasti')
f.salva_figura(fig, 'residuo_per_esemplare', P['figure'])
plt.show()


# --- Figura 5. Gli esiti della rete convolutiva
if len(tabella_cnn):
    fig, assi = plt.subplots(1, len(tabella_cnn), figsize=(6 * len(tabella_cnn), 4),
                             squeeze=False)
    for ax, (_, r) in zip(assi[0], tabella_cnn.iterrows()):
        livelli = ['blocchi', 'registrazioni', 'cuscinetti']
        valori = [r[l] for l in livelli]
        posizioni = np.arange(len(livelli))
        ax.bar(posizioni, valori, 0.5, color=f.COLORI['nostro'], label='rete convolutiva')
        for x, livello in zip(posizioni, livelli):
            ax.hlines(r['degenere_' + livello], x - 0.32, x + 0.32,
                      color=f.COLORI['accento'], lw=2,
                      label='classificatore degenere' if x == 0 else None)
        for x, v in zip(posizioni, valori):
            ax.text(x, v + 1.5, '{:.1f}'.format(v).replace('.', ','),
                    ha='center', fontsize=8)
        ax.set_xticks(posizioni)
        ax.set_xticklabels(['per blocco', 'per registrazione', 'per cuscinetto'],
                           fontsize=8)
        ax.set_ylim(0, 100)
        ax.set_ylabel('accuratezza (%)')
        ax.set_title(r['esperimento'].replace('_', ' '), fontsize=10)
        ax.legend(fontsize=7)
    f.salva_figura(fig, 'esiti_cnn', P['figure'])
    plt.show()

## Salvataggio e materiale per la relazione

Oltre ai CSV, l'ultima cella stampa le due tabelle principali gia in formato `tabular` con
`booktabs` e la virgola decimale, pronte da incollare nel capitolo senza ritrascrivere
numeri a mano.

In [ ]:
def nome_file(testo):
    """Nome di file leggibile: solo lettere, cifre e trattini bassi."""
    pulito = ''.join(c if c.isalnum() else '_' for c in testo)
    while '__' in pulito:
        pulito = pulito.replace('__', '_')
    return pulito.strip('_')


def tabella_latex(df, colonne, intestazioni=None, decimali=3, allineamento=None):
    """Stampa un tabular booktabs con la virgola decimale, pronto per la relazione."""
    intestazioni = intestazioni if intestazioni is not None else colonne

    def formatta(valore):
        if isinstance(valore, (float, np.floating)):
            if np.isnan(valore):
                return '---'
            return ('{:.' + str(decimali) + 'f}').format(valore).replace('.', ',')
        return str(valore).replace('_', ' ').replace('->', '$\\rightarrow$')

    colonne_tex = allineamento or ('l' + 'c' * (len(colonne) - 1))
    print('\\begin{tabular}{' + colonne_tex + '}')
    print('\t\\toprule')
    print('\t' + ' & '.join(intestazioni) + ' \\\\')
    print('\t\\midrule')
    for _, r in df.iterrows():
        print('\t' + ' & '.join(formatta(r[c]) for c in colonne) + ' \\\\')
    print('\t\\bottomrule')
    print('\\end{tabular}')


f.salva_tabella(confronto, 'confronto_esperimenti', P['tabelle'])
f.salva_tabella(tabella_regimi, 'troncamento_per_regime', P['tabelle'])
if len(tabella_cnn):
    f.salva_tabella(tabella_cnn, 'risultati_cnn', P['tabelle'])
for nome, d in dettagli.items():
    f.salva_tabella(d, 'residui_' + nome_file(nome), P['tabelle'])
for nome, e in esiti_cnn.items():
    f.salva_tabella(e['dettaglio'], 'cuscinetti_' + nome_file(nome), P['tabelle'])

print()
print('=' * 70)
print('TABELLA DEI RESIDUI, da incollare nel capitolo')
print('=' * 70)
tabella_latex(confronto,
              ['esperimento', 'residuo_sano', 'rapporto_esterno', 'rapporto_interno',
               'auc_esterno', 'auc_interno', 'auc_media', 'dispersione_esemplari'],
              ['Esperimento', 'residuo sani', 'rapp.\\ esterno', 'rapp.\\ interno',
               'AUC est.', 'AUC int.', 'AUC media', 'dispersione'])

print()
print('=' * 70)
print('TABELLA DEL TRONCAMENTO, da incollare nel capitolo')
print('=' * 70)
tabella_latex(tabella_regimi,
              ['regime', 'frame', 'rms', 'picco',
               'sotto_pavimento_pct', 'sotto_dopo_scala_pct'],
              ['Regime', 'frame', 'valore efficace', 'picco',
               'sotto il pavimento (\\%)', 'dopo la scala (\\%)'])

if len(tabella_cnn):
    print()
    print('=' * 70)
    print('TABELLA DELLA RETE CONVOLUTIVA, da incollare nel capitolo')
    print('=' * 70)
    tabella_latex(tabella_cnn,
                  ['esperimento', 'blocchi', 'degenere_blocchi', 'registrazioni',
                   'cuscinetti', 'degenere_cuscinetti', 'macro_f1', 'macro_f1_degenere'],
                  ['Esperimento', 'per blocco', 'degenere', 'per registrazione',
                   'per cuscinetto', 'degenere', 'macro-$F_1$', 'degenere'],
                  decimali=2)

print()
for cartella in [P['figure'], P['tabelle']]:
    print(cartella)
    for nome in sorted(os.listdir(cartella)):
        percorso = os.path.join(cartella, nome)
        if os.path.isfile(percorso):
            print('  ', nome, round(os.path.getsize(percorso) / 1e6, 3), 'MB')